In [ ]:
import pandas as pd
import numpy as np
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train      = X[X["moon"].isin(splits["train"])]
y_train      = y[y["moon"].isin(splits["train"])]
X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib
import os
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from scipy.stats import pearsonr


def train(X_train, y_train, model_directory_path, loop_moon, embargo):
    feature_cols = [c for c in X_train.columns if c.startswith("Feature_")]

    df = X_train.merge(y_train[["moon", "id", "target"]], on=["moon", "id"])
    df = df.dropna(subset=["target"])
    all_moons = np.array(sorted(df["moon"].unique()))
    n_moons   = len(all_moons)

    # ── Config ────────────────────────────────────────────────────────────────
    K           = 5
    N_QUINTILES = 5
    WINDOW      = 400
    RECENCY_STR = 5.0

    # Restrict to window
    window_moons = all_moons[-WINDOW:].tolist()
    df           = df[df["moon"].isin(window_moons)].copy().reset_index(drop=True)
    all_moons    = np.array(sorted(df["moon"].unique()))
    n_moons      = len(all_moons)
    moon_to_rank = {m: i for i, m in enumerate(all_moons)}
    print(f"  Stacking: {K}-fold LightGBM → Ridge meta-model")
    print(f"  Window: last {WINDOW} moons ({all_moons[0]}→{all_moons[-1]})")

    quintile = (np.arange(n_moons) / n_moons * N_QUINTILES).astype(int)
    quintile = np.clip(quintile, 0, N_QUINTILES - 1)

    # OOF prediction array (one value per training row)
    oof_preds = np.zeros(len(df))
    models    = []
    fold_ics  = []

    for fold in range(K):
        train_idx, val_idx = [], []
        for q in range(N_QUINTILES):
            q_idx  = np.where(quintile == q)[0]
            n_val  = max(1, len(q_idx) // K)
            start  = (fold * n_val) % len(q_idx)
            v_pos  = np.arange(start, start + n_val) % len(q_idx)
            v_mask = np.zeros(len(q_idx), dtype=bool)
            v_mask[v_pos] = True
            val_idx.extend(q_idx[v_mask].tolist())
            train_idx.extend(q_idx[~v_mask].tolist())

        fold_tr_moons  = all_moons[train_idx].tolist()
        fold_val_moons = all_moons[val_idx].tolist()

        tr_row_mask  = df["moon"].isin(fold_tr_moons)
        val_row_mask = df["moon"].isin(fold_val_moons)

        df_tr  = df[tr_row_mask].copy()
        df_val = df[val_row_mask].copy()

        ranks      = df_tr["moon"].map(moon_to_rank).values
        norm_ranks = ranks / max(ranks.max(), 1)
        weights    = np.exp(norm_ranks * RECENCY_STR)

        m = LGBMRegressor(
            n_estimators=300, learning_rate=0.03, num_leaves=63,
            subsample=0.8, colsample_bytree=0.8,
            random_state=fold, n_jobs=-1, verbose=-1,
        )
        m.fit(df_tr[feature_cols].fillna(0), df_tr["target"],
              sample_weight=weights)

        # Fill OOF slot
        val_indices = df.index[val_row_mask].tolist()
        oof_preds[val_indices] = m.predict(df_val[feature_cols].fillna(0))

        corrs = []
        for moon in fold_val_moons:
            grp = df_val[df_val["moon"] == moon]
            if len(grp) < 2: continue
            r, _ = pearsonr(
                oof_preds[df.index[df["moon"] == moon].tolist()],
                grp["target"].values)
            if not np.isnan(r): corrs.append(r)
        ic = float(np.mean(corrs)) if corrs else 0.0
        fold_ics.append(ic)
        models.append(m)
        print(f"    Fold {fold+1}: IC={ic:+.5f}")

    print(f"  Mean OOF IC (L1) = {np.mean(fold_ics):+.5f}")

    # ── Level-2: train Ridge on OOF predictions ───────────────────────────────
    # OOF preds are an unbiased estimate of what each LGBM model predicts
    # on unseen data. Ridge learns to re-weight/calibrate them vs raw target.
    meta_X = oof_preds.reshape(-1, 1)
    meta_y = df["target"].values
    ridge  = Ridge(alpha=1.0)
    ridge.fit(meta_X, meta_y)
    meta_train_r, _ = pearsonr(ridge.predict(meta_X), meta_y)
    print(f"  Ridge meta-model train IC = {meta_train_r:+.5f}")

    joblib.dump({
        "models": models, "ridge": ridge,
        "features": feature_cols,
    }, os.path.join(model_directory_path, "model.joblib"))
    print("  Saved.")


In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):
    obj      = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    models   = obj["models"]
    ridge    = obj["ridge"]
    features = obj["features"]

    X = X_test[features].fillna(0)

    # Average L1 predictions, then pass through Ridge meta-model
    avg_pred = np.mean([m.predict(X) for m in models], axis=0)
    final    = ridge.predict(avg_pred.reshape(-1, 1))

    return pd.DataFrame({
        "moon":       X_test["moon"].values,
        "id":         X_test["id"].values,
        "prediction": final,
    })


In [ ]:
import os
from scipy.stats import pearsonr

embargo   = 4
model_dir = "./model"
os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

results = []
for moon in splits["reduced_cloud"]:
    pred = infer(X_test_cloud[X_test_cloud["moon"] == moon],
                 model_dir, loop_moon=moon, embargo=embargo)
    results.append(pred)

predictions = pd.concat(results)
corrs = []
for moon in splits["reduced_cloud"]:
    merged = predictions[predictions["moon"] == moon].merge(
        y_test_cloud[y_test_cloud["moon"] == moon], on=["moon", "id"])
    if len(merged) > 1:
        r, _ = pearsonr(merged["prediction"], merged["target"])
        corrs.append(r)
        print(f"Moon {moon}: Pearson r = {r:.4f}")
print(f"\nMean IC = {np.mean(corrs):.4f}")
